# Customer Segmentation and Vaue (The RFM Framework)

Let's identify our **most valuable clients** to protect against churn and **target high-ROI** loyalty campaigns.

Core questions to answer:
- Who are our top 10% highest-value customers based on total historical spend?
- What is the distribution of customer purchasing frequency, and how many days on average elapse between repeat orders?
- Can we segment our customer base into distinct tiers (e.g., VIP, Regular, At-Risk) using Recency, Frequency, and Monetary (RFM) metrics?

In [1]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

In [2]:
db_path = Path("..") / "data" / "northwind.db"
engine = create_engine(f"sqlite:///{db_path}")

In [3]:
query = """
    SELECT
        o.CustomerID,
        c.CompanyName,
        o.OrderID,
        o.OrderDate,
        SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS OrderTotal
    FROM Orders o
    JOIN [Order Details] od ON o.OrderID = od.OrderID
    JOIN Customers c ON o.CustomerID = c.CustomerID
    GROUP BY o.CustomerID, c.CompanyName, o.OrderID, o.OrderDate;
"""
df = pd.read_sql_query(query, con=engine)
print("Data ingested successfully!")
print(df.head())

Data ingested successfully!
  CustomerID                CompanyName  OrderID   OrderDate  OrderTotal
0      VINET  Vins et alcools Chevalier    10248  2016-07-04      440.00
1      TOMSP         Toms Spezialitäten    10249  2016-07-05     1863.40
2      HANAR              Hanari Carnes    10250  2016-07-08     1552.60
3      VICTE       Victuailles en stock    10251  2016-07-08      654.06
4      SUPRD           Suprêmes délices    10252  2016-07-09     3597.90


In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16684 entries, 0 to 16683
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CustomerID   16684 non-null  str    
 1   CompanyName  16684 non-null  str    
 2   OrderID      16684 non-null  int64  
 3   OrderDate    16684 non-null  str    
 4   OrderTotal   16684 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 651.8 KB
None


In [7]:
# Change the data type for the OrderDate from string to a date object
df['OrderDate'] = pd.to_datetime(df['OrderDate'], format='mixed')

# Set the reference snapshot date
snapshot_date = df['OrderDate'].max() + pd.Timedelta(days=1)

# Aggregate to customer level
rfm_df = df.groupby(['CustomerID', 'CompanyName']).agg(
    Recency=('OrderDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('OrderID', 'nunique'),
    Monetary=('OrderTotal', 'sum')
).reset_index()

print(rfm_df.head())

  CustomerID                         CompanyName  Recency  Frequency  \
0      ALFKI                 Alfreds Futterkiste       98        184   
1      ANATR  Ana Trujillo Emparedados y helados       23        170   
2      ANTON             Antonio Moreno Taquería        5        170   
3      AROUT                     Around the Horn       22        180   
4      BERGS                  Berglunds snabbköp        8        201   

       Monetary  
0  5.056590e+06  
1  4.588108e+06  
2  4.682125e+06  
3  4.556579e+06  
4  5.538775e+06  


In [9]:
# Sort customers by total spend descending
top_spenders = rfm_df.sort_values(by='Monetary', ascending=False)

# Calculate the top 10% threshold count
top_10_count = int(len(rfm_df) * 0.10)
top_10_customers = top_spenders.head(top_10_count)

print(f"--- Top 10% Highest-Value Customers ({top_10_count} total) ---")
print(top_10_customers[['CustomerID', 'CompanyName', 'Monetary']].to_string(index=False))

--- Top 10% Highest-Value Customers (9 total) ---
CustomerID                CompanyName     Monetary
     QUEDE                Que Delícia 5836778.8600
     VINET  Vins et alcools Chevalier 5823822.8600
     OTTIK         Ottilies Käseladen 5697324.5100
     OCEAN     Océano Atlántico Ltda. 5678533.9200
     RATTC Rattlesnake Canyon Grocery 5639907.3705
     SAVEA         Save-a-lot Markets 5585370.5200
     PICCO           Piccolo und mehr 5577006.8700
     CONSH      Consolidated Holdings 5571519.7500
     BERGS         Berglunds snabbköp 5538774.7875


In [10]:
# Distribution of customer purchasing frequency
print("--- Frequency Distribution Summary ---")
print(rfm_df['Frequency'].describe())

--- Frequency Distribution Summary ---
count     93.000000
mean     179.397849
std       15.313492
min      138.000000
25%      168.000000
50%      178.000000
75%      191.000000
max      219.000000
Name: Frequency, dtype: float64


In [11]:
# Average days between repeat orders per customer
def avg_days_between_orders(group):
    # Sort order dates uniqly for each customer
    dates = group['OrderDate'].drop_duplicates().sort_values()
    if len(dates) < 2: 
        return None
    return dates.diff().dt.days.mean()
    
repeat_intervals = df.groupby('CustomerID').apply(avg_days_between_orders).dropna()

print("\n--- Days Between Repeat Orders Summary ---")
print(f"Overall Average: {repeat_intervals.mean():.1f} days")
print(repeat_intervals.describe())


--- Days Between Repeat Orders Summary ---
Overall Average: 28.6 days
count    93.000000
mean     28.575669
std       2.474808
min      23.348624
25%      26.689840
50%      28.477528
75%      30.467066
max      37.218978
dtype: float64


In [12]:
"""
To answer the final question, whether we can segment our customers based on distinct tiers,
we can score our customers using quartiles and group them into actional segments. 
"""
# Assign quartile scores (1 to 4)
# For Recency: lower days since last order is better, so ascending=False for ranking/bins or reverse labels
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], 4, labels=[4, 3, 2, 1])
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], 4, labels=[1, 2, 3, 4])

# Combine individual scores into an RFM score string
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

# Define a function to segment customers into clear tiers
def segment_customer(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    if f >= 3 and m >= 3 and r >= 3:
        return 'VIP / Champion'
    elif r <= 2 and m >= rfm_df['Monetary'].median():
        return 'At-Risk High Value'
    else:
        return 'Regular / Standard'

rfm_df['Segment'] = rfm_df.apply(segment_customer, axis=1)

# Display segment counts and a sample of the results
print("--- Customer Segment Distribution ---")
print(rfm_df['Segment'].value_counts())
print("\n--- Sample Segmented Data ---")
print(rfm_df[['CustomerID', 'CompanyName', 'Recency', 'Frequency', 'Monetary', 'Segment']].head(10).to_string(index=False))

--- Customer Segment Distribution ---
Segment
Regular / Standard    71
VIP / Champion        22
Name: count, dtype: int64

--- Sample Segmented Data ---
CustomerID                        CompanyName  Recency  Frequency     Monetary            Segment
     ALFKI                Alfreds Futterkiste       98        184 5056590.5000 Regular / Standard
     ANATR Ana Trujillo Emparedados y helados       23        170 4588108.2700 Regular / Standard
     ANTON            Antonio Moreno Taquería        5        170 4682124.6375 Regular / Standard
     AROUT                    Around the Horn       22        180 4556578.5900 Regular / Standard
     BERGS                 Berglunds snabbköp        8        201 5538774.7875     VIP / Champion
     BLAUS            Blauer See Delikatessen        7        177 4942889.6900 Regular / Standard
     BLONP           Blondesddsl père et fils       78        181 5299582.5400 Regular / Standard
     BOLID          Bólido Comidas preparadas       30        1

## Findings & Results Summary

### 1. Top 10% Highest-Value Customers (Monetary)

Our high-value spenders drive a significant portion of total revenue. The top 10% of customers ranked by historical spend include:

1. Que Delícia ($5,836,778.86)
2. Vins et alcools Chevalier ($5,823,822.86)
3. Ottilies Käseladen ($5,697,324.51)
4. Océano Atlántico Ltda. ($5,678,533.92)
5. Rattlesnake Canyon Grocery ($5,639,907.37)
6. Save-a-lot Markets ($5,585,370.52)
7. Piccolo und mehr ($5,577,006.87)
8. Consolidated Holdings ($5,571,519.75)
9. Berglunds snabbköp ($5,538,774.79)

---

### 2. Purchasing Frequency & Repeat Order Intervals

* **Frequency Distribution:** Customer purchase counts across the dataset range from a minimum of **138 orders** to a maximum of **219 orders**, with a median of **178 orders** per customer.
* **Repeat Order Intervals:** On average, repeat orders across the customer base happen every **28.6 days** (ranging between 23.3 and 37.2 days depending on the individual client).

---

### 3. RFM Customer Segmentation Tiers

Using quartile rankings for Recency, Frequency, and Monetary metrics, the customer base was categorized into actionable segments:

* **VIP / Champion:** 22 customers (High engagement, recent activity, and top-tier spend)
* **Regular / Standard:** 71 customers (Steady base requiring targeted nurturing to prevent churn)